# XAI - Relative importance mass (Analysis 2) - TP vs TN, all targets

For each patient and each signature `k`, the `grad_score` of the **top-K patches** are summed into a
30-component vector, then normalised: `v_norm = v / sum(v)` (relative importance, comparable across
patients).

**Hypothesis.** `v_norm[target]` is higher when the model **detects the signature correctly**.
**Primary contrast TP vs TN** (one-sided MWU, TP>TN), **secondary plus vs minus** (biological
presence). Status = group (signature in the true top-3) x eligible (signature in the predicted top-3).

**Score.** Raw `grad_score` (mass). `grad_specific` only as a robustness check (cell 6).
Runs on the 4 pooled folds (per-tissue pooled cache).


## 0. Config

In [ ]:
import os, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu
import pyarrow.parquet as pq
from tqdm.auto import tqdm

GRAD_DIR  = '../grad'
OUT_DIR   = os.path.join(GRAD_DIR, 'xai_outputs')
SPLIT_DIR = '../coad_stad_explainability'
FOLD_DIR  = '../grad_cam_signatures_tcga'   # parquet grad-score per fold (per costruire le cache mancanti)
os.makedirs(OUT_DIR, exist_ok=True)

SCORE_COL   = 'grad_score'
RANK_SCORES = ['grad_score', 'combined_score', 'grad_specific']  # combined = attn*ReLU(grad)
K_LIST      = [3, 5, 10, 20, 50, 100]
SEED        = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

USECOLS  = ['sample_id', 'case_id', 'signature_name', 'patch_index', 'grad_score', 'attention_weight']
CAT_COLS = ['sample_id', 'case_id', 'signature_name']

colors = {'TP': '#2ca02c', 'FN': '#ff7f0e', 'TN': '#d62728', 'FP': '#9467bd'}
# primario = TP vs TN (modello correttamente presente vs assente); secondario = plus vs minus
GROUP_DEF = {'plus': {'TP', 'FN'}, 'minus': {'TN', 'FP'},
             'pred': {'TP', 'FP'}, 'nopred': {'FN', 'TN'},   # decisione del modello (include FP)
             'TP': {'TP'}, 'FN': {'FN'}, 'TN': {'TN'}, 'FP': {'FP'}}
CONTRASTS = [('TP', 'TN'), ('pred', 'TN')]   # TP vs TN (modello corretto) + (TP+FP) vs TN (decisione del modello, incl. FP)
COMBINED_ABS = True   # True: combined = attn*|grad| ; False: attn*ReLU(grad)
EXPLAIN_FILE = os.path.join(SPLIT_DIR, 'explain_top5.parquet')   # stato TP/FN/TN/FP; toggle: explain_top5.parquet
import re as _re
_m = _re.search(r'top\d+', os.path.basename(EXPLAIN_FILE))
TAG = ('_' + _m.group(0)) if _m else ''   # es '_top3' -> appeso a tutti gli output
TAG = TAG + ('_absC' if COMBINED_ABS else '')   # output separati
print('TAG output =', TAG or '(nessuno)')
EXPLAIN = pd.read_parquet(EXPLAIN_FILE)
EXPLAINS = {v: pd.read_parquet(os.path.join(SPLIT_DIR, f'explain_{v}.parquet')) for v in ['top3', 'top5']}
VERSIONS = list(EXPLAINS)   # ogni test viene fatto per entrambe le versioni
# ALL tumors and signatures present in the file (dynamic, no longer hardcoded)
TISSUES = {tt.lower(): {'targets': sorted(EXPLAIN.loc[EXPLAIN['tumor_type'] == tt, 'signature'].unique())}
           for tt in sorted(EXPLAIN['tumor_type'].unique())}
print('explain:', os.path.basename(EXPLAIN_FILE), EXPLAIN.shape, '| tumori:', len(TISSUES), '| contrasti:', CONTRASTS)
print({t: c['targets'] for t, c in TISSUES.items()})

## 1. Helper: cache subset per tessuto, preparazione score, vettori di importanza

In [ ]:
def build_cache(tissue):
    '''Costruisce la cache pooled del tessuto dai parquet grad-score dei 4 fold (una slide alla volta).'''
    import glob
    cs = set(EXPLAIN.loc[EXPLAIN['tumor_type'].str.upper() == tissue.upper(), 'Case ID'])
    parqs = sorted(glob.glob(os.path.join(FOLD_DIR, 'val_gradcam_fold[1-4]_*.parquet')))
    out = os.path.join(OUT_DIR, f'attr_{tissue}_subset_scores.parquet'); writer = None; nsl = 0
    for pp in parqs:
        pf = pq.ParquetFile(pp)
        for rg in range(pf.metadata.num_row_groups):
            cid = pf.read_row_group(rg, columns=['case_id']).column('case_id')[0].as_py()
            if cid in cs:
                t = pf.read_row_group(rg)
                if writer is None:
                    writer = pq.ParquetWriter(out, t.schema, compression='snappy')
                writer.write_table(t); nsl += 1
    if writer is not None:
        writer.close()
    print(f'[{tissue}] cache costruita: {nsl} slide da {len(parqs)} fold parquet')

def split_path(tissue, target):
    return os.path.join(SPLIT_DIR, f'{tissue}_{target}{TAG}.csv')

def load_subset(tissue):
    '''Read the tissue parquet cache: USECOLS only, strings as category (lower memory).'''
    pth = os.path.join(OUT_DIR, f'attr_{tissue}_subset_scores.parquet')
    if not os.path.exists(pth):
        build_cache(tissue)
    names = pq.ParquetFile(pth).schema_arrow.names
    cols = [c for c in USECOLS if c in names]
    tbl = pq.read_table(pth, columns=cols)
    for c in CAT_COLS:
        if c in tbl.column_names:
            i = tbl.schema.get_field_index(c)
            tbl = tbl.set_column(i, c, tbl.column(c).dictionary_encode())
    a = tbl.to_pandas()
    print(f'[{tissue}] {os.path.basename(pth)}: {a.shape}  mem={a.memory_usage(deep=True).sum()/1e9:.2f} GB')
    return a

def prep(a):
    '''combined_score = attention * ReLU(grad) (needs the signed gradient); then abs(grad) = raw mass.'''
    g = a['grad_score'].astype('float32')
    a['combined_score'] = (a['attention_weight'].astype('float32') *
                               (np.abs(g) if COMBINED_ABS else np.clip(g, 0, None))).astype('float32')
    a['grad_score'] = np.abs(g)
    return a

def add_grad_specific(a):
    '''grad_specific = grad - mean across signatures on the same (slide, patch). Robustness check only.'''
    if 'grad_specific' not in a.columns:
        gp = a.groupby(['sample_id', 'patch_index'], observed=True)['grad_score']
        a['grad_specific'] = (a['grad_score'] - gp.transform('mean')).astype('float32')
    return a

def importance_vectors(attr, score, K):
    '''Sum the top-K scores of each signature per slide, sum the patient's slides, then normalise.'''
    case_of = attr.groupby('sample_id', observed=True)['case_id'].first().astype(str)
    per = []
    for sid, g in attr.groupby('sample_id', observed=True):
        per.append(g.groupby('signature_name', observed=True)[score]
                   .apply(lambda s: s.nlargest(K).sum()).rename(str(sid)))
    V = pd.DataFrame(per)
    V['case_id'] = [case_of.get(s) for s in V.index]
    Vc = V.groupby('case_id').sum(numeric_only=True).clip(lower=0)
    return Vc.div(Vc.sum(axis=1), axis=0)

def build_all_missing_caches():
    '''Build the missing caches for ALL tumors in ONE pass (routing each slide to its tumor).'''
    import glob
    tumor_of = dict(zip(EXPLAIN['Case ID'], EXPLAIN['tumor_type'].str.lower()))
    missing = [t for t in TISSUES if not os.path.exists(os.path.join(OUT_DIR, f'attr_{t}_subset_scores.parquet'))]
    if not missing:
        print('cache: tutte gia presenti'); return
    print('costruisco cache per', len(missing), 'tumori:', missing, flush=True)
    writers = {}
    for pp in sorted(glob.glob(os.path.join(FOLD_DIR, 'val_gradcam_fold[1-4]_*.parquet'))):
        pf = pq.ParquetFile(pp)
        for rg in range(pf.metadata.num_row_groups):
            cid = pf.read_row_group(rg, columns=['case_id']).column('case_id')[0].as_py()
            t = tumor_of.get(cid)
            if t in missing:
                tbl = pf.read_row_group(rg)
                if t not in writers:
                    writers[t] = pq.ParquetWriter(os.path.join(OUT_DIR, f'attr_{t}_subset_scores.parquet'),
                                                  tbl.schema, compression='snappy')
                writers[t].write_table(tbl)
    for w in writers.values():
        w.close()
    print('cache costruite:', sorted(writers.keys()), flush=True)

def labels_for(tissue, target, ver=None):
    '''From explain_topN.parquet (one row per case,signature): group=plus if the signature is in true_topN,
    eligible=yes se in pred_topN. Stato: plus&yes=TP, plus&no=FN, minus&yes=FP (falsi positivi), minus&no=TN.'''
    ver = ver if ver is not None else VERSIONS[-1]
    ex = EXPLAINS[ver]
    sub = ex[(ex['tumor_type'].str.upper() == tissue.upper()) & (ex['signature'] == target)]
    present = dict(zip(sub['Case ID'], np.where(sub['group'] == 'plus', 'present', 'absent')))
    status = {}
    for cid, g_, el in zip(sub['Case ID'], sub['group'], sub['eligible']):
        if g_ == 'plus':
            status[cid] = 'TP' if el == 'yes' else 'FN'
        else:
            status[cid] = 'FP' if el == 'yes' else 'TN'
    return present, status

def status_of(tissue, target, index, ver=None):
    _, status = labels_for(tissue, target, ver)
    return pd.Series({c: status.get(c) for c in index})

def grp(s, st, label):
    return s[st.isin(GROUP_DEF[label])].dropna()   # scarta NaN (gruppi vuoti / v_norm indefinito)

def box_contrast(ax, s, st, gA, gB, ylabel, alt):
    a = grp(s, st, gA); b = grp(s, st, gB)
    p = mannwhitneyu(a, b, alternative=alt).pvalue if len(a) and len(b) else np.nan
    try:
        bp = ax.boxplot([a.values, b.values], tick_labels=[gA, gB], widths=0.5,
                        patch_artist=True, showfliers=False)
    except TypeError:
        bp = ax.boxplot([a.values, b.values], labels=[gA, gB], widths=0.5,
                        patch_artist=True, showfliers=False)
    for patch, col in zip(bp['boxes'], ['#8fd19e', '#f1a6a6']):
        patch.set_facecolor(col); patch.set_alpha(0.6)
    for i, lab in enumerate([gA, gB], 1):
        sub = grp(s, st, lab)
        ax.scatter(rng.normal(i, 0.06, len(sub)), sub.values,
                   c=[colors[st[c]] for c in sub.index], s=28, zorder=3, edgecolor='k', linewidth=0.3)
    ax.set_ylabel(ylabel)
    return a, b, p

## 2. Load/build the subsets and compute `Vn` for COAD and STAD
`Vn[tissue][K]` = matrix (cases x 30 signatures) of relative importance, independent of the target.
The first run on COAD scans the large CSV, after which it is cached.


In [ ]:
build_all_missing_caches()
DATA = {}
for tissue in list(TISSUES):
    try:
        attr = prep(load_subset(tissue))
    except Exception as e:
        print(f'[{tissue}] SKIP (nessuna attribuzione): {type(e).__name__}'); TISSUES.pop(tissue, None); continue
    Vn = {K: importance_vectors(attr, SCORE_COL, K) for K in K_LIST}
    DATA[tissue] = {'attr': attr, 'Vn': Vn}
    print(f'[{tissue}] casi={attr["case_id"].nunique()} slide={attr["sample_id"].nunique()} '
          f'targets={len(TISSUES[tissue]["targets"])}  Vn[100]={Vn[100].shape}', flush=True)

## 2b. TP only — no comparison against TN
A purely descriptive view: for each (tissue, signature) it takes `v_norm[target]` **on the TP cases
only** (the signature is both true and correctly predicted), at every K. No statistical test, no
TN/FN/FP groups: it just shows where the importance concentrates when the model gets it right,
without any contrast. Saves `vnorm_TP_only_ALL{TAG}.csv`.


In [ ]:
rows_tp_only = []
for tissue in TISSUES:
    Vn = DATA[tissue]['Vn']
    for target in TISSUES[tissue]['targets']:
        ks, series = [], []
        for K in K_LIST:
            if target not in Vn[K].columns:
                continue
            s = Vn[K][target]
            st = status_of(tissue, target, s.index)
            tp = grp(s, st, 'TP')
            if len(tp) == 0:
                continue
            ks.append(K); series.append(tp)
            rows_tp_only.append({'tissue': tissue, 'target': target, 'K': K, 'n_TP': len(tp),
                                 'mean': round(tp.mean(), 4), 'median': round(tp.median(), 4),
                                 'std': round(tp.std(), 4) if len(tp) > 1 else 0.0})
        if not series:
            continue
        fig, ax = plt.subplots(figsize=(1.3 * len(ks) + 2, 4))
        bp = ax.boxplot([tp.values for tp in series], tick_labels=[str(k) for k in ks],
                        widths=0.5, patch_artist=True, showfliers=False)
        for patch in bp['boxes']:
            patch.set_facecolor(colors['TP']); patch.set_alpha(0.6)
        for i, tp in enumerate(series, 1):
            ax.scatter(rng.normal(i, 0.06, len(tp)), tp.values, c=colors['TP'], s=28,
                      zorder=3, edgecolor='k', linewidth=0.3)
        ax.set_xlabel('K'); ax.set_ylabel(f'v_norm[{target}] (solo TP)')
        ax.set_title(f'{tissue.upper()} - {target}: v_norm nei soli TP (n varia con K)')
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f'{tissue}_{target}_vnorm_TPonly{TAG}.png'), dpi=150, bbox_inches='tight')
        plt.show()
tp_only_summary = pd.DataFrame(rows_tp_only)
tp_only_summary.to_csv(os.path.join(OUT_DIR, f'vnorm_TP_only_ALL{TAG}.csv'), index=False)
tp_only_summary


## 3. `v_norm[target]`: **TP vs TN** - all targets
For each target, a TP vs TN boxplot at the various K (points coloured by status) and a one-sided MWU
(TP>TN). `vnorm_present_vs_absent_ALL.csv` reports both TP-vs-TN and plus-vs-minus.


In [ ]:
gA, gB = CONTRASTS[0]   # TP vs TN
all_rows = []
for tissue in TISSUES:
    Vn = DATA[tissue]['Vn']
    for target in TISSUES[tissue]['targets']:
        fig, axes = plt.subplots(1, len(K_LIST), figsize=(5 * len(K_LIST), 4.5), squeeze=False)
        for j, K in enumerate(K_LIST):
            if target not in Vn[K].columns:
                continue
            s = Vn[K][target]; st = status_of(tissue, target, s.index)
            _, _, p = box_contrast(axes[0][j], s, st, gA, gB, f'v_norm[{target}]', alt='greater')
            axes[0][j].set_title(f'K={K} - p({gA}>{gB})={p:.3g}')
            for ver in VERSIONS:
                stv = status_of(tissue, target, s.index, ver)
                for (cA, cB) in CONTRASTS:
                    x = grp(s, stv, cA); z = grp(s, stv, cB)
                    pv = mannwhitneyu(x, z, alternative='greater').pvalue if len(x) and len(z) else np.nan
                    all_rows.append({'version': ver, 'tissue': tissue, 'target': target, 'K': K,
                                     'contrast': f'{cA}>{cB}', 'nA': len(x), 'nB': len(z),
                                     'meanA': round(x.mean(), 4), 'meanB': round(z.mean(), 4),
                                     'delta': round(x.mean() - z.mean(), 4), 'MWU_p': round(pv, 4)})
        axes[0][-1].legend(handles=[Line2D([0],[0], marker='o', ls='', mec='k', mfc=colors[s2], label=s2)
                                    for s2 in ['TP','FN','TN','FP']], title='stato', fontsize=7)
        fig.suptitle(f'{tissue.upper()} - {target}: v_norm {gA} vs {gB}', y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f'{tissue}_{target}_vnorm_TPvsTN{TAG}.png'), dpi=150, bbox_inches='tight')
        plt.show()
summary = pd.DataFrame(all_rows)
summary.to_csv(os.path.join(OUT_DIR, f'vnorm_present_vs_absent_ALL{TAG}.csv'), index=False)
summary

## 4. Profilo di importanza medio (30 componenti) - TP vs TN
`v_norm` medio nei due gruppi (top-100): la componente del target dovrebbe alzarsi nei TP.

In [ ]:
K = K_LIST[-1]; gA, gB = CONTRASTS[0]
for tissue in TISSUES:
    Vn = DATA[tissue]['Vn']
    for target in TISSUES[tissue]['targets']:
        if target not in Vn[K].columns:
            continue
        st = status_of(tissue, target, Vn[K].index)
        A = Vn[K][st.isin(GROUP_DEF[gA])]; B = Vn[K][st.isin(GROUP_DEF[gB])]
        comp = pd.DataFrame({gA: A.mean(), gB: B.mean()})
        comp['delta'] = comp[gA] - comp[gB]
        comp = comp.sort_values(gA, ascending=False)
        fig, ax = plt.subplots(figsize=(8, max(4, 0.30 * len(comp))))
        y = np.arange(len(comp)); h = 0.4
        cp = ['#2ca02c' if s == target else '#8fd19e' for s in comp.index]
        ca = ['#d62728' if s == target else '#f1a6a6' for s in comp.index]
        ax.barh(y + h/2, comp[gA], height=h, color=cp, label=gA)
        ax.barh(y - h/2, comp[gB], height=h, color=ca, label=gB)
        ax.set_yticks(y); ax.set_yticklabels(comp.index, fontsize=7); ax.invert_yaxis()
        ax.set_xlabel(f'v_norm medio (top-{K}, {SCORE_COL})'); ax.legend()
        ax.set_title(f'{tissue.upper()} - importanza relativa media ({target} evidenziata, {gA} vs {gB})')
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f'{tissue}_{target}_vnorm_profile{TAG}.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print(f'[{tissue} {target}] delta v_norm[{target}] ({gA}-{gB}) = {comp.loc[target, "delta"]:+.4f}')

## 5. Per-tissue panel: `v_norm[target]` TP vs TN across all targets
One plot per tissue (top-100). TP > TN means the model weights the signature more when it detects it.


In [ ]:
K = K_LIST[-1]; gA, gB = CONTRASTS[0]
panels = {}
for tissue in TISSUES:
    Vn = DATA[tissue]['Vn']
    rows = []
    for target in TISSUES[tissue]['targets']:
        if target not in Vn[K].columns:
            continue
        st = status_of(tissue, target, Vn[K].index); s = Vn[K][target]
        a = grp(s, st, gA); b = grp(s, st, gB)
        p = mannwhitneyu(a, b, alternative='greater').pvalue if len(a) and len(b) else np.nan
        rows.append({'target': target, 'nA': len(a), 'nB': len(b), 'meanA': round(a.mean(), 4),
                     'meanB': round(b.mean(), 4), 'delta': round(a.mean() - b.mean(), 4), 'MWU_p': round(p, 4)})
    panel = pd.DataFrame(rows); panels[tissue] = panel
    panel.to_csv(os.path.join(OUT_DIR, f'{tissue}_vnorm_panel_{SCORE_COL}_K{K}{TAG}.csv'), index=False)
    fig, ax = plt.subplots(figsize=(max(7, 2 * len(panel)), 5))
    x = np.arange(len(panel)); w = 0.38
    ax.bar(x - w/2, panel['meanA'], w, color='#2ca02c', alpha=0.8, label=gA)
    ax.bar(x + w/2, panel['meanB'], w, color='#d62728', alpha=0.8, label=gB)
    for i, r in panel.iterrows():
        ax.text(i, max(r['meanA'], r['meanB']), f"p={r['MWU_p']}\nn={r['nA']}/{r['nB']}",
                ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(panel['target'])
    ax.set_ylabel(f'v_norm[target] medio (top-{K})')
    ax.set_title(f'{tissue.upper()}: massa di importanza {gA} vs {gB}\n({gA} > {gB} = il modello pesa la firma quando la rileva)')
    ax.legend(); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'{tissue}_vnorm_panel_{SCORE_COL}{TAG}.png'), dpi=150, bbox_inches='tight')
    plt.show()
panel_all = pd.concat({t: p.set_index('target') for t, p in panels.items()}, names=['tissue'])
panel_all.to_csv(os.path.join(OUT_DIR, f'vnorm_panel_ALL_{SCORE_COL}_K{K}{TAG}.csv'))
panel_all

## 6. Robustness: grad_score vs **combined_score** (= attn * ReLU(grad)) vs contrastive
For each (tissue, target) the test is repeated with the three rankings at every K and on **both
contrasts** (TP>TN and plus>minus). `combined_score` weights the gradient by the model's attention.
Saved to `vnorm_robustness_ALL.csv`.


In [ ]:
rows = []
for tissue in TISSUES:
    attr = add_grad_specific(DATA[tissue]['attr'])
    for score in RANK_SCORES:
        Vk_by_K = {K: importance_vectors(attr, score, K) for K in K_LIST}
        for target in TISSUES[tissue]['targets']:
            for K in K_LIST:
                Vk = Vk_by_K[K]
                if target not in Vk.columns:
                    continue
                s = Vk[target]
                for ver in VERSIONS:
                    st = status_of(tissue, target, s.index, ver)
                    for (cA, cB) in CONTRASTS:
                        a = grp(s, st, cA); b = grp(s, st, cB)
                        p = mannwhitneyu(a, b, alternative='greater').pvalue if len(a) and len(b) else np.nan
                        rows.append({'version': ver, 'tissue': tissue, 'target': target, 'score': score, 'K': K,
                                     'contrast': f'{cA}>{cB}', 'nA': len(a), 'nB': len(b),
                                     'meanA': round(a.mean(), 4), 'meanB': round(b.mean(), 4),
                                     'delta': round(a.mean() - b.mean(), 4), 'MWU_p': round(p, 4)})
robust = pd.DataFrame(rows)
robust.to_csv(os.path.join(OUT_DIR, f'vnorm_robustness_ALL{TAG}.csv'), index=False)
robust[robust['K'] == 100]

## 6. Reading the result
- **TP vs TN** isolates the model's behaviour when it is consistent (correctly detecting vs correctly
  excluding), removing FN/FP. `v_norm` is normalised per patient, so TP>TN is not a magnitude
  artefact but a relative share.
- **plus vs minus** (in the table) is the biological comparison, more conservative.
- **Caveats.** Some TP groups are small (SBS93, SBS8), so the p-values are unstable and a
  multiple-comparison correction is needed. Cell 6 checks raw against contrastive.


## 7. Significant combinations: **p < 0.054** (~0.05) — STANDALONE cell
Re-reads `vnorm_robustness_ALL.csv` from disk, so it works on a fresh kernel, and shows every
combination (tumor, signature, score, K, contrast) with `MWU_p < 0.054`, sorted by p. Rows where one
group has fewer than `NMIN` cases are flagged as unreliable, which is typical where the TN group
collapses. Saves `vnorm_significativi_p05.csv`.


In [ ]:
# ---- STANDALONE: ALL combinations with p ~<= 0.05 (notebook 2) ----
import os
import numpy as np, pandas as pd
from IPython.display import display

OUT_DIR_ = globals().get('OUT_DIR', '../grad/xai_outputs')
import re as _re
TAG = globals().get('TAG')
if TAG is None:
    _ef = globals().get('EXPLAIN_FILE', 'explain_top3.parquet')
    _m = _re.search(r'top\d+', os.path.basename(str(_ef)))
    TAG = ('_' + _m.group(0)) if _m else ''
print('TAG =', TAG or '(nessuno)')
CSV      = os.path.join(OUT_DIR_, f'vnorm_robustness_ALL{TAG}.csv')
NMIN     = 10   # 'affidabile' = ENTRAMBI i gruppi >= NMIN pazienti (flag, non filtra)

src = pd.read_csv(CSV)
src['MWU_p'] = pd.to_numeric(src['MWU_p'], errors='coerce')
print('letto:', CSV, '->', src.shape, '| tumori:', sorted(src['tissue'].unique()))

hits = src[src['MWU_p'].round(2) <= 0.05].copy()      # 0.054 arrotonda a 0.05 -> incluso
hits['n_min'] = hits[['nA', 'nB']].min(axis=1)
hits['affidabile'] = np.where(hits['n_min'] >= NMIN, 'si', 'NO')
hits = hits.sort_values('MWU_p').reset_index(drop=True)

print(f'\ncombinazioni con p arrotondato <= 0.05: {len(hits)} su {len(src)}'
      f'   | affidabili: {(hits["affidabile"]=="si").sum()}   | gruppo piccolo: {(hits["affidabile"]=="NO").sum()}')

cols = ['tissue', 'target', 'score', 'K', 'contrast', 'nA', 'nB', 'delta', 'MWU_p', 'n_min', 'affidabile']
pd.set_option('display.max_rows', 500)
print('\n================= TUTTE le combinazioni (ordinate per p) =================')
display(hits[cols])

out = os.path.join(OUT_DIR_, f'vnorm_significativi_p05{TAG}.csv')
hits.to_csv(out, index=False)
print('\nsalvato:', out)

## 8. From significant signatures to the **important patches** (cropped from the WSI)
For a chosen `(TISSUE, TARGET)`: takes the **TP** patients, extracts their **top-N patches** by the
given score and **crops them from the WSI** (openslide) using `coord_x`/`coord_y`, plotting them in a
grid.

**WSI note**: locally available slides are **TCGA-READ** (`CMSProject/dataset*`) and CPTAC
(`external_dataset`). For the other tumors the `.svs` is not present, so the cell **falls back** to
showing the **spatial map** of the top patches (coordinates) instead of the images.


In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

TISSUE     = 'read'             # tumore (chiave minuscola, es. 'read', 'coad', 'brca')
TARGET     = 'SBS93'           # firma
SCORE      = 'combined_score'   # 'grad_score' oppure 'combined_score'
N_CASES    = 3                  # quanti pazienti TP mostrare
N_PATCH    = 6                  # quante patch per paziente
PATCH_20X  = 224                # patch NEL FRAME 20x (== estrazione)
DISPLAY    = 256                # dimensione di visualizzazione (px)

OUT_DIR_  = globals().get('OUT_DIR', '../grad/xai_outputs')
import re as _re
TAG = globals().get('TAG')
if TAG is None:
    _ef = globals().get('EXPLAIN_FILE', 'explain_top3.parquet')
    _m = _re.search(r'top\d+', os.path.basename(str(_ef)))
    TAG = ('_' + _m.group(0)) if _m else ''
print('TAG =', TAG or '(nessuno)')
EXPL_     = globals().get('EXPLAIN_FILE', '../coad_stad_explainability/explain_top3.parquet')
# NB: coordinates are in the 20x frame; GDC .svs are 40x -> map with coord*factor (factor=AppMag/20)
WSI_DIRS  = ['../gdc', '../external_dataset', '../../CMSProject/dataset', '../../CMSProject/dataset_focus']

try:
    import openslide; HAVE_OS = True
except Exception:
    HAVE_OS = False; print('openslide non disponibile -> uso il fallback mappa spaziale')

def find_wsi(sample_id):
    for d in WSI_DIRS:
        hit = glob.glob(os.path.join(d, str(sample_id) + '.svs')) + \
              glob.glob(os.path.join(d, '**', str(sample_id) + '.svs'), recursive=True)
        if hit:
            return hit[0]
    return None

def _mag(wsi):
    for k in ('aperio.AppMag', openslide.PROPERTY_NAME_OBJECTIVE_POWER):
        v = wsi.properties.get(k)
        if v:
            try: return float(v)
            except ValueError: pass
    return None

def _crop(wsi, x, y, factor):
    p0 = PATCH_20X * factor
    W, H = wsi.dimensions
    ox = int(np.clip(x * factor, 0, max(0, W - p0))); oy = int(np.clip(y * factor, 0, max(0, H - p0)))
    return wsi.read_region((ox, oy), 0, (p0, p0)).convert('RGB').resize((DISPLAY, DISPLAY))

def _pick_factor(wsi, top, mag):
    W, H = wsi.dimensions
    cand = [max(1, int(round(mag / 20)))] if mag else [2, 1]
    for f in ([2, 1] if not mag else cand + [2, 1]):
        ok = all(int(r['coord_x']) * f + PATCH_20X * f <= W and
                 int(r['coord_y']) * f + PATCH_20X * f <= H for _, r in top.iterrows())
        if ok:
            return f
    return 1

# 1) stato TP/FN/TN/FP dal file explain
ex = pd.read_parquet(EXPL_)
sub = ex[(ex['tumor_type'].str.upper() == TISSUE.upper()) & (ex['signature'] == TARGET)]
status = {cid: ('TP' if (g == 'plus' and e == 'yes') else 'FN' if g == 'plus'
                else 'FP' if e == 'yes' else 'TN')
          for cid, g, e in zip(sub['Case ID'], sub['group'], sub['eligible'])}
tp_cases = [c for c, s in status.items() if s == 'TP']
print(f'{TISSUE} {TARGET}: TP={len(tp_cases)}')

# 2) attribuzione + coordinate per quella firma
pth = os.path.join(OUT_DIR_, f'attr_{TISSUE}_subset_scores.parquet')
d = pq.read_table(pth, columns=['sample_id', 'case_id', 'signature_name',
                                'coord_x', 'coord_y', 'grad_score', 'attention_weight']).to_pandas()
d = d[d['signature_name'].astype(str) == TARGET].copy()
g = d['grad_score'].astype('float32')
d['combined_score'] = (d['attention_weight'].astype('float32') *
                       (np.abs(g) if COMBINED_ABS else np.clip(g, 0, None))).astype('float32')
d['grad_score'] = np.abs(g)
d['case_id'] = d['case_id'].astype(str); d['sample_id'] = d['sample_id'].astype(str)

cases = [c for c in tp_cases if c in set(d['case_id'])][:N_CASES]
print('pazienti TP con attribuzione:', cases)

# 3) for each case: top-N patches -> crop from the WSI (or spatial map)
for case in cases:
    dc = d[d['case_id'] == case]
    sid = dc['sample_id'].iloc[0]
    top = dc.nlargest(N_PATCH, SCORE)
    wsi = find_wsi(sid)
    if HAVE_OS and wsi:
        slide = openslide.open_slide(wsi)
        mag = _mag(slide); factor = _pick_factor(slide, top, mag)
        print(f'  {case}: {os.path.basename(wsi)} AppMag={mag} factor={factor}')
        fig, axes = plt.subplots(1, len(top), figsize=(2.2 * len(top), 2.6))
        for ax, (_, r) in zip(np.atleast_1d(axes), top.iterrows()):
            ax.imshow(_crop(slide, int(r['coord_x']), int(r['coord_y']), factor)); ax.axis('off')
            ax.set_title(f"{SCORE[:8]}={r[SCORE]:.2e}", fontsize=7)
        fig.suptitle(f'{TISSUE.upper()} {TARGET} - {case} (TP): top-{N_PATCH} patch per {SCORE}', y=1.05)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_, f'patches_{TISSUE}_{TARGET}_{case}{TAG}.png'), dpi=150, bbox_inches='tight')
        plt.show()
    else:
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(dc['coord_x'], dc['coord_y'], s=3, c='#cccccc', label='tutte le patch')
        ax.scatter(top['coord_x'], top['coord_y'], s=40, c='#d62728', label=f'top-{N_PATCH}')
        ax.invert_yaxis(); ax.set_aspect('equal'); ax.legend(fontsize=8)
        ax.set_title(f'{TISSUE.upper()} {TARGET} - {case} (TP)\nWSI non trovata: mappa spaziale dei top-patch', fontsize=9)
        plt.tight_layout(); plt.show()
        print(f'  (slide {sid}.svs non trovata in {WSI_DIRS})')

## 8b. Patches of the 12 selected TPs — SBS8, SBS12, SBS39, SBS18 (**top-3** criterion, TP only)
STANDALONE cell: it does not depend on the `TISSUE`/`TARGET` of the previous cell.

It reads `../gdc/xai_tp_slides.csv` directly: the patients are **TP under the top-3 criterion**
(signature both true **and** predicted within the first 3, not the first 5) — no comparison against
TN, no requirement for a negative counterpart. For each one it crops the top-N patches by
`combined_score` from the WSI in `../gdc/<uuid>/`.


In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

SCORE_PATCH = 'combined_score'   # 'grad_score' oppure 'combined_score'
N_PATCH     = 8                   # quante patch per paziente
PATCH_20X   = 224                 # patch NEL FRAME 20x (== estrazione)
DISPLAY     = 256                 # dimensione di visualizzazione (px)
WSI_DIRS_TP = ['../gdc', '../external_dataset']   # coord in frame 20x; svs GDC 40x -> factor=AppMag/20

try:
    import openslide; HAVE_OS_TP = True
except Exception:
    HAVE_OS_TP = False; print('openslide non disponibile -> skip patch (nessun fallback qui)')

def find_wsi_tp(sample_id):
    import glob
    for d in WSI_DIRS_TP:
        hit = glob.glob(os.path.join(d, '**', str(sample_id) + '.svs'), recursive=True)
        if hit:
            return hit[0]
    return None

def mag_tp(wsi):
    for k in ('aperio.AppMag', openslide.PROPERTY_NAME_OBJECTIVE_POWER):
        v = wsi.properties.get(k)
        if v:
            try: return float(v)
            except ValueError: pass
    return None

def pick_factor_tp(wsi, top, mag):
    W, H = wsi.dimensions
    cand = [max(1, int(round(mag / 20)))] if mag else [2, 1]
    for f in ([2, 1] if not mag else cand + [2, 1]):
        ok = all(int(r['coord_x']) * f + PATCH_20X * f <= W and
                 int(r['coord_y']) * f + PATCH_20X * f <= H for _, r in top.iterrows())
        if ok:
            return f
    return 1

def crop_tp(wsi, x, y, factor):
    p0 = PATCH_20X * factor
    W, H = wsi.dimensions
    ox = int(np.clip(x * factor, 0, max(0, W - p0))); oy = int(np.clip(y * factor, 0, max(0, H - p0)))
    return wsi.read_region((ox, oy), 0, (p0, p0)).convert('RGB').resize((DISPLAY, DISPLAY))

xai_sel = pd.read_csv('../gdc/xai_tp_slides.csv')
xai_sel['tissue'] = xai_sel['tumor_type'].str.lower()
print(xai_sel[['signature', 'tumor_type', 'case_id']].to_string(index=False))

# reuse/build the per-tissue caches (same logic as section 2); attr holds case_id, signature_name, coord_x/y
DATA_TP = {}
for tissue in xai_sel['tissue'].unique():
    try:
        DATA_TP[tissue] = prep(load_subset(tissue)) if tissue not in DATA else DATA[tissue]['attr']
    except Exception as e:
        print(f'[{tissue}] impossibile costruire/leggere la cache: {type(e).__name__}: {e}'); continue

if not HAVE_OS_TP:
    raise SystemExit('serve openslide per ritagliare le patch')

for _, row in xai_sel.iterrows():
    tissue, target, case = row['tissue'], row['signature'], row['case_id']
    attr = DATA_TP.get(tissue)
    if attr is None:
        continue
    dc = attr[(attr['case_id'].astype(str) == case) & (attr['signature_name'].astype(str) == target)]
    if dc.empty:
        print(f'[{tissue} {target} {case}] nessuna riga di attribuzione in cache -> skip'); continue
    sid = dc['sample_id'].iloc[0]
    top = dc.nlargest(N_PATCH, SCORE_PATCH)
    wsi_path = find_wsi_tp(sid)
    if not wsi_path:
        print(f'[{tissue} {target} {case}] WSI {sid}.svs non trovata sotto {WSI_DIRS_TP} -> skip'); continue
    slide = openslide.open_slide(wsi_path)
    mag = mag_tp(slide); factor = pick_factor_tp(slide, top, mag)
    fig, axes = plt.subplots(1, len(top), figsize=(2.2 * len(top), 2.6))
    for ax, (_, r) in zip(np.atleast_1d(axes), top.iterrows()):
        ax.imshow(crop_tp(slide, int(r['coord_x']), int(r['coord_y']), factor)); ax.axis('off')
        ax.set_title(f"{SCORE_PATCH[:8]}={r[SCORE_PATCH]:.2e}", fontsize=7)
    fig.suptitle(f'{tissue.upper()} {target} - {case} (TP top-3): top-{N_PATCH} patch per {SCORE_PATCH}', y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'patches_TPtop3_{tissue}_{target}_{case}.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 8. **Which** combinations are significant, and at which K
Grid (tumor, signature, score) x K: a **green** cell means the rounded p <= 0.05 (so 0.054 is
included), grey means not significant; the p is annotated in the lit cells. Includes both
**`combined_score` and `grad_score`**. Rows are sorted by best p. Below, a table listing at which K
each combination comes out.


In [ ]:
# ---- STANDALONE: QUALI combinazioni significative e a quali K (per versione top3/top5) ----
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR_   = globals().get('OUT_DIR', '../grad/xai_outputs')
import re as _re
TAG = globals().get('TAG')
if TAG is None:
    _ef = globals().get('EXPLAIN_FILE', 'explain_top3.parquet')
    _m = _re.search(r'top\d+', os.path.basename(str(_ef)))
    TAG = ('_' + _m.group(0)) if _m else ''
print('TAG =', TAG or '(nessuno)')
CSV        = os.path.join(OUT_DIR_, f'vnorm_robustness_ALL{TAG}.csv')
SCORES_SEL = ['combined_score', 'grad_score']

d_all = pd.read_csv(CSV)
d_all['MWU_p'] = pd.to_numeric(d_all['MWU_p'], errors='coerce')
d_all = d_all[d_all['score'].isin(SCORES_SEL)].copy()
VERS = sorted(d_all['version'].unique()) if 'version' in d_all.columns else [None]
BASE = _re.sub(r'_top\d+', '', TAG)   # toglie _topN dal TAG, tiene _absC
print('versioni nel file:', VERS)

for VER in VERS:
    d = d_all.copy() if VER is None else d_all[d_all['version'] == VER].copy()
    vtag = TAG if VER is None else f'{BASE}_{VER}'
    vlbl = '' if VER is None else f' [{VER}]'
    d['sig'] = d['MWU_p'].round(2) <= 0.05
    d['lab'] = d['tissue'].str.upper() + ' ' + d['target'] + '  [' + d['score'].str.replace('_score', '', regex=False) + ']'
    Ks = sorted(d['K'].dropna().unique())
    print(f'\n######## versione {VER} ######## score:', SCORES_SEL, '| K:', [int(k) for k in Ks])

    for contrast, sub in d.groupby('contrast'):
        keep = sorted(sub.loc[sub['sig'], 'lab'].unique())
        if not keep:
            print(f'[{contrast}{vlbl}] nessuna combinazione significativa'); continue
        S  = sub.pivot_table(index='lab', columns='K', values='sig',   aggfunc='max').reindex(keep)
        PV = sub.pivot_table(index='lab', columns='K', values='MWU_p', aggfunc='min').reindex(keep)
        order = PV.min(axis=1).sort_values().index
        S, PV = S.loc[order], PV.loc[order]
        fig, ax = plt.subplots(figsize=(1.5 * len(Ks) + 6, max(3, 0.34 * len(order))))
        ax.imshow(S.fillna(False).values.astype(float), aspect='auto',
                  cmap=plt.matplotlib.colors.ListedColormap(['#eeeeee', '#2ca02c']), vmin=0, vmax=1)
        for i in range(S.shape[0]):
            for j in range(S.shape[1]):
                if bool(S.fillna(False).values[i, j]):
                    ax.text(j, i, f'{PV.values[i, j]:.3f}', ha='center', va='center', fontsize=6.5, color='w')
        ax.set_xticks(range(len(S.columns))); ax.set_xticklabels([f'K={int(k)}' for k in S.columns])
        ax.set_yticks(range(len(order)));     ax.set_yticklabels(order, fontsize=7)
        ax.set_title(f'{contrast}{vlbl} - combinazioni significative (p arrotondato <= 0.05), per K', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_, f'quali_sig_per_K_{contrast.replace(">", "gt")}{vtag}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        tab = pd.DataFrame({
            'quali_K': [', '.join(str(int(k)) for k in S.columns[S.loc[l].fillna(False).values]) for l in order],
            'p_min':   PV.min(axis=1).round(4)})
        print(f'\n===== {contrast}{vlbl} =====')
        pd.set_option('display.max_rows', 300)
        display(tab)


## 9. Tumor x signature grid — **shape = K**, score = `combined_score`
Same style as notebook 1, applied to the mass (`v_norm`): **(B)** a single panel with four
sub-columns per tumor (one per K, each with a different shape); **(F)** faceted, one panel per K.
Colour = -log10(p). STANDALONE cell: it re-reads `vnorm_robustness_ALL.csv`.


In [ ]:
# ---- STANDALONE: tumor x signature grid (v_norm) — for EVERY score, contrast and version ----
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUT_DIR_  = globals().get('OUT_DIR', '../grad/xai_outputs')
import re as _re
TAG = globals().get('TAG')
if TAG is None:
    _ef = globals().get('EXPLAIN_FILE', 'explain_top3.parquet')
    _m = _re.search(r'top\d+', os.path.basename(str(_ef)))
    TAG = ('_' + _m.group(0)) if _m else ''
print('TAG =', TAG or '(nessuno)')
CSV        = os.path.join(OUT_DIR_, f'vnorm_robustness_ALL{TAG}.csv')
SCORES_SEL   = ['combined_score', 'grad_score']   # tutti gli score da graficare (aggiungi 'grad_specific' se vuoi)
CONTRASTS_SEL = ['TP>TN', 'pred>TN']              # tutti i contrasti

d_all = pd.read_csv(CSV)
d_all['MWU_p'] = pd.to_numeric(d_all['MWU_p'], errors='coerce')
VERS = sorted(d_all['version'].unique()) if 'version' in d_all.columns else [None]
BASE = _re.sub(r'_top\d+', '', TAG)
_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', 'h', 'p']
plt.rcParams.update({'font.size': 9})
print('versioni:', VERS, '| score:', SCORES_SEL, '| contrasti:', CONTRASTS_SEL)

def _draw(dv, title, fbase):
    dv = dv[dv['MWU_p'].round(2) <= 0.05].copy()
    if dv.empty:
        print('  (nessuna significativa) ', title); return
    dv['lvl'] = dv['K'].astype(int).astype(str)
    lvls = [str(int(k)) for k in sorted(dv['K'].dropna().unique())]
    dv['negp'] = -np.log10(dv['MWU_p'].clip(lower=1e-6))
    tums = sorted(dv['tissue'].str.upper().unique())
    sigs = dv.groupby('target').size().sort_values().index.tolist()
    xi = {t: i for i, t in enumerate(tums)}; yi = {s: i for i, s in enumerate(sigs)}
    mk = {l: _MARKERS[i % len(_MARKERS)] for i, l in enumerate(lvls)}
    vmin, vmax = 0.0, float(dv['negp'].max()); cmap = plt.cm.viridis

    def _scatter(ax, sub, spread=True):
        for l in lvls:
            s2 = sub[sub['lvl'] == l]
            if s2.empty: continue
            off = (lvls.index(l) - (len(lvls) - 1) / 2) * (0.66 / max(len(lvls), 1)) if spread else 0
            ax.scatter(s2['tissue'].str.upper().map(xi) + off, s2['target'].map(yi),
                       c=s2['negp'], cmap=cmap, vmin=vmin, vmax=vmax, marker=mk[l],
                       s=55, edgecolor='0.25', linewidth=0.5, zorder=3)
        ax.set_xticks(range(len(tums))); ax.set_xticklabels(tums, rotation=90, fontsize=7)
        ax.set_yticks(range(len(sigs))); ax.set_yticklabels(sigs, fontsize=7)
        ax.set_xlim(-0.7, len(tums) - 0.3); ax.set_ylim(-0.8, len(sigs) - 0.2)
        ax.grid(color='0.92', lw=0.5, zorder=0)
        for s_ in ('top', 'right'): ax.spines[s_].set_visible(False)

    # Layout B
    fig, ax = plt.subplots(figsize=(0.55 * len(tums) + 5, 0.30 * len(sigs) + 2.5))
    _scatter(ax, dv)
    ax.set_title(title + '\nshape = n. patch (K)', fontsize=11)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
    plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.01).set_label('-log10(p-value)')
    ax.legend(handles=[Line2D([], [], marker=mk[l], ls='', color='0.35', markersize=7, label=f'K = {l}')
                       for l in lvls], title='n. patch (K)', loc='upper left',
              bbox_to_anchor=(1.06, 1), frameon=False, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR_, f'vnorm_griglia_B_{fbase}.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    # Layout sfaccettato per K
    ncol = 2; nrow = int(np.ceil(len(lvls) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * (0.45 * len(tums) + 3.2),
                                                  nrow * (0.26 * len(sigs) + 2.2)), squeeze=False)
    for a, l in zip(axes.flat, lvls):
        _scatter(a, dv[dv['lvl'] == l], spread=False)
        a.set_title(f'K = {l}', fontsize=11)
    for a in axes.flat[len(lvls):]: a.axis('off')
    fig.suptitle(title + ' — sfaccettato per numero di patch', fontsize=13, y=1.0)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
    fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.015, pad=0.015).set_label('-log10(p-value)')
    plt.savefig(os.path.join(OUT_DIR_, f'vnorm_griglia_facet_{fbase}.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'  OK {title}: {len(dv)} punti | {dv["tissue"].nunique()} tumori | {dv["target"].nunique()} firme')

for VER in VERS:
    dV = d_all.copy() if VER is None else d_all[d_all['version'] == VER].copy()
    vtag = TAG if VER is None else f'{BASE}_{VER}'
    vlbl = '' if VER is None else f' [{VER}]'
    for SCORE_SEL in SCORES_SEL:
        for CONTRAST in CONTRASTS_SEL:
            dv = dV[(dV['score'] == SCORE_SEL) & (dV['contrast'] == CONTRAST)]
            title = f'Massa v_norm ({SCORE_SEL}){vlbl} — firme significative per tumore, contrasto {CONTRAST}'
            fbase = f'{SCORE_SEL}_{CONTRAST.replace(">","gt")}{vtag}'
            _draw(dv, title, fbase)
